# 📊 Live NSE Stock Dashboard — Google Colab + ngrok
### Real-time stock data served as a beautiful web dashboard with a live public URL

**How it works:**
1. Run **Cell 1** → installs packages
2. Run **Cell 2** → paste your ngrok token (free at ngrok.com)
3. Run **Cell 3** → launches the Flask server
4. Run **Cell 4** → prints your **live public URL** — share it with anyone!

> The dashboard auto-refreshes every 30 seconds with live yfinance data.


## Cell 1 — Install Packages

In [5]:
!pip install yfinance flask pyngrok requests --quiet
print("✅ All packages installed.")

✅ All packages installed.


## Cell 2 — Set ngrok Auth Token

1. Go to **https://dashboard.ngrok.com/signup** (free account)
2. Copy your authtoken from **https://dashboard.ngrok.com/get-started/your-authtoken**
3. Paste it below and run this cell


In [6]:
from pyngrok import ngrok

# ✏️ PASTE YOUR NGROK TOKEN HERE
NGROK_TOKEN = "3CjEXjaMFS2dKVwl8PdzQpssadr_76ZvXjcZx7jBcwrveCYAP"

ngrok.set_auth_token(NGROK_TOKEN)
print("✅ ngrok token set.")

✅ ngrok token set.


## Cell 3 — Launch the Dashboard Server

In [7]:
import threading, json, time, requests
import yfinance as yf
import numpy as np
import pandas as pd
from flask import Flask, jsonify, render_template_string

app = Flask(__name__)

STOCKS = {
    "RELIANCE":       "RELIANCE.NS",
    "TCS":            "TCS.NS",
    "HDFC BANK":      "HDFCBANK.NS",
    "INFOSYS":        "INFY.NS",
    "MOTILAL OSWAL":  "MOTILALOFS.NS",
    "BANK OF BARODA": "BANKBARODA.NS",
    "GAIL":           "GAIL.NS",
    "VEDANTA":        "VEDL.NS",
    "BIRLASOFT":      "BSOFT.NS",
    "NMDC":           "NMDC.NS",
}

def get_usd_to_inr():
    try:
        r = requests.get("https://api.exchangerate-api.com/v4/latest/USD", timeout=5)
        return r.json()["rates"]["INR"]
    except:
        return 83.5

def classify_mcap(crores):
    if crores is None: return "N/A"
    if crores > 20_000: return "Large Cap"
    if crores > 5_000:  return "Mid Cap"
    return "Small Cap"

# ── NaN-safe helpers ──────────────────────────────────────────
import math as _math

def _sf(v, decimals=2, fallback="N/A"):
    """Return clean rounded float, or fallback if NaN/None/inf."""
    try:
        f = float(v)
        return fallback if (_math.isnan(f) or _math.isinf(f)) else round(f, decimals)
    except (TypeError, ValueError):
        return fallback

def _clean_list(lst):
    """Replace NaN/inf in a list with None (→ JSON null)."""
    out = []
    for v in lst:
        if v is None:
            out.append(None)
        else:
            try:
                f = float(v)
                out.append(None if (_math.isnan(f) or _math.isinf(f)) else round(f, 2))
            except (TypeError, ValueError):
                out.append(None)
    return out
# ──────────────────────────────────────────────────────────────

def _fmt_cr(val):
    """Convert raw INR value → crores string, or N/A."""
    try:
        v = float(val)
        if _math.isnan(v) or _math.isinf(v): return "N/A"
        return f"{round(v/1e7):,}"
    except: return "N/A"

def _get_balance_sheet(ticker):
    try:
        bs = ticker.balance_sheet
        if bs is None or bs.empty: return {}
        bs = bs.iloc[:, 0]  # most recent year
        def g(keys):
            for k in keys:
                for idx in bs.index:
                    if k.lower() in str(idx).lower():
                        return _fmt_cr(bs[idx])
            return "N/A"
        return {
            "Total Assets":        g(["Total Assets"]),
            "Total Liabilities":   g(["Total Liabilities Net Minority Interest","Total Liabilities"]),
            "Total Equity":        g(["Stockholders Equity","Total Equity"]),
            "Cash & Equivalents":  g(["Cash And Cash Equivalents","Cash Cash Equivalents"]),
            "Total Debt":          g(["Total Debt","Long Term Debt"]),
            "Current Assets":      g(["Current Assets","Total Current Assets"]),
            "Current Liabilities": g(["Current Liabilities","Total Current Liabilities"]),
            "Retained Earnings":   g(["Retained Earnings"]),
        }
    except: return {}

def _get_financials(ticker):
    try:
        inc = ticker.quarterly_financials
        if inc is None or inc.empty: return {"quarters":[], "revenue":[], "net_income":[], "ebitda":[]}
        inc = inc.iloc[:, :4]  # last 4 quarters
        quarters = [str(c)[:10] for c in inc.columns]
        def row(keys):
            for k in keys:
                for idx in inc.index:
                    if k.lower() in str(idx).lower():
                        return [_fmt_cr(inc[c][idx]) for c in inc.columns]
            return ["N/A"]*len(quarters)
        return {
            "quarters":   quarters,
            "revenue":    row(["Total Revenue","Revenue"]),
            "net_income": row(["Net Income","Net Income Common Stockholders"]),
            "ebitda":     row(["EBITDA","Normalized EBITDA"]),
            "gross_profit": row(["Gross Profit"]),
        }
    except: return {"quarters":[], "revenue":[], "net_income":[], "ebitda":[], "gross_profit":[]}

def _get_shareholding(inf):
    try:
        inst  = _sf(inf.get("heldPercentInstitutions", 0), decimals=4, fallback=0)
        ins_pct = round((inst if isinstance(inst, float) else 0) * 100, 2)
        promo = _sf(inf.get("heldPercentInsiders", 0), decimals=4, fallback=0)
        pro_pct = round((promo if isinstance(promo, float) else 0) * 100, 2)
        public_pct = round(max(0, 100 - ins_pct - pro_pct), 2)
        return {
            "Promoters":       pro_pct,
            "Institutions":    ins_pct,
            "Public & Others": public_pct,
        }
    except: return {}

def _get_corp_actions(ticker):
    actions = []
    try:
        divs = ticker.dividends
        if hasattr(divs.index, "tz") and divs.index.tz:
            divs.index = divs.index.tz_localize(None)
        for dt, amt in divs[-8:].items():
            actions.append({"date": str(dt)[:10], "type": "Dividend", "value": f"₹{round(float(amt),2)}"})
    except: pass
    try:
        splits = ticker.splits
        if hasattr(splits.index, "tz") and splits.index.tz:
            splits.index = splits.index.tz_localize(None)
        for dt, ratio in splits[-4:].items():
            actions.append({"date": str(dt)[:10], "type": "Split", "value": f"{ratio}:1"})
    except: pass
    actions.sort(key=lambda x: x["date"], reverse=True)
    return actions[:12]

def get_stock_data(symbol):
    try:
        t   = yf.Ticker(symbol)
        inf = t.info
        df  = t.history(period="1y", interval="1d")
        if df.empty:
            df = yf.download(symbol, period="1y", interval="1d", progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        if hasattr(df.index, "tz") and df.index.tz:
            df.index = df.index.tz_localize(None)
        df = df.dropna(subset=["Close"])

        price  = df["Close"]  if "Close"  in df.columns else pd.Series(dtype=float)
        volume = df["Volume"] if "Volume" in df.columns else pd.Series(dtype=float)

        # RSI
        delta = price.diff()
        gain  = delta.clip(lower=0).rolling(14).mean()
        loss  = (-delta.clip(upper=0)).rolling(14).mean()
        rs    = gain / loss.replace(0, np.nan)
        rsi   = (100 - (100 / (1 + rs))).dropna()

        # MAs
        sma20 = price.rolling(20).mean()
        sma50 = price.rolling(50).mean()

        # 52W — guard against NaN from rolling on short series
        _h52_raw = price.rolling(252).max().iloc[-1] if len(price) >= 10 else float("nan")
        _l52_raw = price.rolling(252).min().iloc[-1] if len(price) >= 10 else float("nan")
        h52  = None if pd.isna(_h52_raw) else float(_h52_raw)
        l52  = None if pd.isna(_l52_raw) else float(_l52_raw)
        curr = float(price.iloc[-1]) if not price.empty else None
        pct52 = round((curr - l52) / (h52 - l52) * 100, 1)                 if (curr and h52 and l52 and h52 != l52) else 50

        # Market Cap — yfinance returns INR for .NS; no FX conversion needed
        fx       = get_usd_to_inr()
        mcap_inr = inf.get("marketCap")
        mcap_cr  = round(mcap_inr / 1e7) if mcap_inr else None

        # Price history for chart
        dates  = [d.strftime("%d %b") for d in price.index[-60:]]
        prices = _clean_list(price.values[-60:])
        sma20v = _clean_list(sma20.values[-60:])
        sma50v = _clean_list(sma50.values[-60:])
        vols   = _clean_list([v / 1e6 for v in volume.values[-60:]])
        rsi_d  = [d.strftime("%d %b") for d in rsi.index[-60:]]
        rsi_v  = _clean_list(rsi.values[-60:])

        prev    = float(price.iloc[-2]) if len(price) > 1 else curr
        chg     = _sf(curr - prev) if (curr is not None and prev) else 0
        chg_pct = _sf(chg / prev * 100) if prev else 0

        # Dividends
        divs = t.dividends
        if hasattr(divs.index, "tz") and divs.index.tz:
            divs.index = divs.index.tz_localize(None)
        div_yield = inf.get("dividendYield") or 0
        last_div  = _sf(divs.iloc[-1]) if not divs.empty else 0

        def _inf(key):
            return _sf(inf.get(key)) if inf.get(key) is not None else "N/A"

        return {
            "name":         inf.get("longName", symbol),
            "symbol":       symbol,
            "price":        _sf(curr) if curr is not None else "N/A",
            "change":       chg,
            "change_pct":   chg_pct,
            "high":         _sf(df["High"].iloc[-1])  if "High" in df.columns else "N/A",
            "low":          _sf(df["Low"].iloc[-1])   if "Low"  in df.columns else "N/A",
            "prev_close":   _sf(prev),
            "mcap_cr":      f"{mcap_cr:,}" if mcap_cr else "N/A",
            "mcap_cls":     classify_mcap(mcap_cr),
            "pe":           _inf("trailingPE"),
            "pb":           _inf("priceToBook"),
            "eps":          _inf("trailingEps"),
            "beta":         _inf("beta"),
            "div_yield":    _sf(div_yield * 100, decimals=2, fallback=0),
            "last_div":     last_div,
            "sector":       inf.get("sector", "N/A"),
            "industry":     inf.get("industry", "N/A"),
            "h52":          _sf(h52) if h52 else "N/A",
            "l52":          _sf(l52) if l52 else "N/A",
            "pct52":        pct52,
            "rsi":          _sf(rsi.iloc[-1], 1) if not rsi.empty else "N/A",
            "avg_vol":      _sf(volume.mean() / 1e6) if not volume.empty else "N/A",
            "chart_dates":  dates,
            "chart_prices": prices,
            "chart_sma20":  sma20v,
            "chart_sma50":  sma50v,
            "chart_vols":   vols,
            "rsi_dates":    rsi_d,
            "rsi_values":   rsi_v,
            "summary":      (inf.get("longBusinessSummary") or "")[:400] + "...",
            "fx_rate":      _sf(fx),

            # ── Balance Sheet (latest annual) ──
            "balance_sheet": _get_balance_sheet(t),

            # ── Income Statement (last 4 quarters) ──
            "financials":    _get_financials(t),

            # ── Shareholding Pattern ──
            "shareholding":  _get_shareholding(inf),

            # ── Corporate Actions ──
            "corp_actions":  _get_corp_actions(t),
        }
    except Exception as e:
        return {"error": str(e)}


HTML = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>NSE Live Dashboard</title>
<link href="https://fonts.googleapis.com/css2?family=DM+Mono:wght@400;500&family=Syne:wght@400;600;700;800&display=swap" rel="stylesheet">
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js"></script>
<style>
:root {
  --bg: #0a0e1a;
  --card: #111827;
  --card2: #161d2e;
  --border: #1e2d45;
  --accent: #00d4ff;
  --accent2: #7c3aed;
  --green: #10b981;
  --red: #ef4444;
  --text: #e2e8f0;
  --muted: #64748b;
  --gold: #f59e0b;
}
* { margin:0; padding:0; box-sizing:border-box; }
body { background:var(--bg); color:var(--text); font-family:"Syne",sans-serif;
       min-height:100vh; }

/* Header */
.header { background:linear-gradient(135deg,#0a0e1a 0%,#111827 100%);
  border-bottom:1px solid var(--border); padding:18px 32px;
  display:flex; align-items:center; justify-content:space-between;
  position:sticky; top:0; z-index:100; backdrop-filter:blur(12px); }
.logo { font-size:1.3rem; font-weight:800; letter-spacing:-0.5px;
  background:linear-gradient(90deg,var(--accent),var(--accent2));
  -webkit-background-clip:text; -webkit-text-fill-color:transparent; }
.live-badge { display:flex; align-items:center; gap:8px; font-size:.78rem;
  color:var(--green); font-family:"DM Mono",monospace; }
.pulse { width:8px; height:8px; background:var(--green); border-radius:50%;
  animation:pulse 1.5s infinite; }
@keyframes pulse { 0%,100%{opacity:1;transform:scale(1)}
  50%{opacity:.4;transform:scale(1.4)} }

/* Stock selector tabs */
.selector { padding:16px 32px; display:flex; gap:8px; flex-wrap:wrap;
  border-bottom:1px solid var(--border); background:var(--card); }
.tab { padding:6px 16px; border-radius:20px; border:1px solid var(--border);
  background:transparent; color:var(--muted); cursor:pointer;
  font-family:"Syne",sans-serif; font-size:.82rem; font-weight:600;
  transition:all .2s; letter-spacing:.5px; }
.tab:hover { border-color:var(--accent); color:var(--accent); }
.tab.active { background:var(--accent); color:#000; border-color:var(--accent);
  font-weight:700; }

/* Main layout */
.main { padding:24px 32px; display:grid;
  grid-template-columns:1fr 1fr 1fr; gap:20px; max-width:1600px; margin:0 auto; }
@media(max-width:1100px){.main{grid-template-columns:1fr 1fr;}}
@media(max-width:700px){.main{grid-template-columns:1fr;padding:16px;}}

/* Hero price card */
.hero { grid-column:1/-1; background:var(--card);
  border:1px solid var(--border); border-radius:16px; padding:28px 32px;
  display:flex; align-items:center; justify-content:space-between;
  flex-wrap:wrap; gap:20px;
  background:linear-gradient(135deg,#111827 60%,#0d1929 100%);
  position:relative; overflow:hidden; }
.hero::before { content:""; position:absolute; top:-40px; right:-40px;
  width:200px; height:200px; background:radial-gradient(circle,rgba(0,212,255,.07),transparent);
  border-radius:50%; }
.hero-left .company-name { font-size:1.6rem; font-weight:800; margin-bottom:4px; }
.hero-left .symbol { color:var(--muted); font-family:"DM Mono",monospace;
  font-size:.85rem; }
.hero-price { font-size:3rem; font-weight:800; letter-spacing:-2px;
  font-family:"DM Mono",monospace; color:var(--accent); }
.hero-chg { font-size:1.1rem; font-weight:600; margin-top:4px; }
.hero-chg.up { color:var(--green); }
.hero-chg.dn { color:var(--red); }
.hero-stats { display:flex; gap:32px; flex-wrap:wrap; }
.hero-stat label { color:var(--muted); font-size:.72rem; text-transform:uppercase;
  letter-spacing:1px; display:block; margin-bottom:4px; }
.hero-stat span { font-family:"DM Mono",monospace; font-size:1rem; font-weight:500; }
.badge { display:inline-block; padding:3px 12px; border-radius:20px;
  font-size:.72rem; font-weight:700; letter-spacing:.5px; }
.badge-large { background:rgba(0,212,255,.15); color:var(--accent); }
.badge-mid   { background:rgba(245,158,11,.15); color:var(--gold); }
.badge-small { background:rgba(239,68,68,.15);  color:var(--red); }

/* Cards */
.card { background:var(--card); border:1px solid var(--border);
  border-radius:14px; padding:22px; }
.card.span2 { grid-column:span 2; }
@media(max-width:700px){.card.span2{grid-column:span 1;}}
.card-title { font-size:.72rem; text-transform:uppercase; letter-spacing:2px;
  color:var(--muted); margin-bottom:16px; font-weight:700; }

/* Metric grid */
.metrics { display:grid; grid-template-columns:1fr 1fr; gap:14px; }
.metric { background:var(--card2); border-radius:10px; padding:14px 16px;
  border:1px solid var(--border); }
.metric label { font-size:.68rem; color:var(--muted); text-transform:uppercase;
  letter-spacing:1px; display:block; margin-bottom:6px; }
.metric span { font-family:"DM Mono",monospace; font-size:1.1rem; font-weight:500;
  color:var(--text); }

/* RSI meter */
.rsi-bar { height:10px; background:#1e2d45; border-radius:6px; margin:10px 0;
  position:relative; overflow:hidden; }
.rsi-fill { height:100%; border-radius:6px;
  transition:width .6s ease; }
.rsi-labels { display:flex; justify-content:space-between;
  font-size:.68rem; color:var(--muted); font-family:"DM Mono",monospace; }

/* 52W gauge */
.gauge-bar { height:12px; background:#1e2d45; border-radius:6px; margin:12px 0;
  position:relative; }
.gauge-fill { height:100%; border-radius:6px; background:linear-gradient(90deg,var(--red),var(--gold),var(--green));
  transition:width .6s ease; }
.gauge-marker { position:absolute; top:-4px; width:4px; height:20px;
  background:#fff; border-radius:2px; transform:translateX(-50%);
  transition:left .6s ease; }
.gauge-labels { display:flex; justify-content:space-between;
  font-size:.68rem; color:var(--muted); font-family:"DM Mono",monospace; }

/* Scorecard */
.scorecard { display:flex; flex-direction:column; gap:10px; }
.score-item { display:flex; align-items:center; justify-content:space-between;
  padding:10px 14px; background:var(--card2); border-radius:8px;
  border:1px solid var(--border); }
.score-label { font-size:.82rem; font-weight:600; }
.score-val { font-size:.72rem; font-weight:700; padding:2px 10px;
  border-radius:12px; }
.score-high { background:rgba(16,185,129,.15); color:var(--green); }
.score-low  { background:rgba(239,68,68,.15);  color:var(--red); }
.score-mid  { background:rgba(245,158,11,.15); color:var(--gold); }

/* Charts */
.chart-wrap { position:relative; height:220px; }
.chart-wrap-sm { position:relative; height:160px; }

/* Peer table */
.peer-table { width:100%; border-collapse:collapse; font-size:.82rem; }
.peer-table th { color:var(--muted); text-transform:uppercase; letter-spacing:1px;
  font-size:.68rem; padding:8px 10px; text-align:left;
  border-bottom:1px solid var(--border); }
.peer-table td { padding:10px 10px; border-bottom:1px solid rgba(30,45,69,.5); }
.peer-table tr:hover td { background:var(--card2); }
.up-text { color:var(--green); font-family:"DM Mono",monospace; }
.dn-text { color:var(--red);   font-family:"DM Mono",monospace; }

/* Dividend */
.div-row { display:flex; justify-content:space-between; align-items:center;
  padding:10px 0; border-bottom:1px solid rgba(30,45,69,.5); }
.div-row:last-child { border:none; }

/* Loader */
.loader { display:flex; flex-direction:column; align-items:center;
  justify-content:center; height:60vh; gap:16px; }
.spinner { width:44px; height:44px; border:3px solid var(--border);
  border-top-color:var(--accent); border-radius:50%;
  animation:spin .8s linear infinite; }
@keyframes spin{to{transform:rotate(360deg)}}
.loader p { color:var(--muted); font-size:.9rem; font-family:"DM Mono",monospace; }

/* refresh */
.refresh-bar { text-align:center; padding:8px; font-size:.72rem;
  color:var(--muted); font-family:"DM Mono",monospace;
  border-top:1px solid var(--border); margin-top:8px; }

.summary-text { font-size:.82rem; line-height:1.7; color:#94a3b8; }

/* Section Tabs */
.sec-tabs { display:flex; gap:6px; margin-bottom:18px; flex-wrap:wrap; }
.sec-tab { padding:5px 14px; border-radius:20px; border:1px solid var(--border);
  background:transparent; color:var(--muted); cursor:pointer;
  font-family:"Syne",sans-serif; font-size:.78rem; font-weight:600;
  transition:all .2s; letter-spacing:.3px; }
.sec-tab:hover { border-color:var(--accent); color:var(--accent); }
.sec-tab.active { background:var(--accent2); color:#fff; border-color:var(--accent2); }
.sec-panel { display:none; }
.sec-panel.active { display:contents; }

/* Balance Sheet table */
.bs-table { width:100%; border-collapse:collapse; font-size:.82rem; }
.bs-table td { padding:9px 12px; border-bottom:1px solid var(--border); }
.bs-table td:first-child { color:var(--muted); font-weight:600; }
.bs-table td:last-child { font-family:"DM Mono",monospace; color:var(--text); text-align:right; }
.bs-table tr:last-child td { border-bottom:none; }

/* Financials bar chart */
.fin-bar-wrap { display:flex; flex-direction:column; gap:10px; margin-top:8px; }
.fin-row { display:flex; align-items:center; gap:10px; font-size:.78rem; }
.fin-label { color:var(--muted); width:80px; flex-shrink:0; font-size:.72rem; }
.fin-bar-bg { flex:1; height:18px; background:#1e2d45; border-radius:4px; overflow:hidden; }
.fin-bar-fill { height:100%; border-radius:4px; transition:width .6s ease; }
.fin-val { font-family:"DM Mono",monospace; color:var(--text); font-size:.72rem; width:80px; text-align:right; flex-shrink:0; }

/* Corporate actions table */
.ca-table { width:100%; border-collapse:collapse; font-size:.82rem; }
.ca-table th { padding:8px 12px; text-align:left; color:var(--muted); font-size:.68rem;
  text-transform:uppercase; letter-spacing:1px; border-bottom:1px solid var(--border); font-weight:700; }
.ca-table td { padding:9px 12px; border-bottom:1px solid rgba(30,45,69,.5); }
.ca-table tr:last-child td { border-bottom:none; }
.ca-badge { display:inline-block; padding:2px 10px; border-radius:10px; font-size:.68rem; font-weight:700; }
.ca-div  { background:rgba(16,185,129,.15); color:#10b981; }
.ca-split{ background:rgba(0,212,255,.15); color:#00d4ff; }
.ca-bonus{ background:rgba(245,158,11,.15); color:#f59e0b; }

/* Loading spinner for extra */
.mini-loader { text-align:center; padding:24px; color:var(--muted); font-size:.82rem; }
</style>
</head>
<body>

<div class="header">
  <div class="logo">⚡ NSE Live Dashboard</div>
  <div style="display:flex;gap:20px;align-items:center">
    <span id="fxRate" style="font-size:.75rem;color:var(--muted);font-family:'DM Mono',monospace"></span>
    <div class="live-badge"><div class="pulse"></div>LIVE</div>
  </div>
</div>

<div class="selector" id="tabs"></div>

<div id="app"><div class="loader"><div class="spinner"></div><p>Fetching live data...</p></div></div>

<script>
const STOCKS = {{ stocks_json|safe }};
let currentStock = Object.keys(STOCKS)[3]; // INFOSYS default
let priceChart, rsiChart, volChart;

function buildTabs(){
  const c = document.getElementById("tabs");
  Object.keys(STOCKS).forEach(name=>{
    const b = document.createElement("button");
    b.className = "tab" + (name===currentStock?" active":"");
    b.textContent = name;
    b.onclick=()=>{ currentStock=name;
      document.querySelectorAll(".tab").forEach(x=>x.classList.remove("active"));
      b.classList.add("active");
      loadData();
    };
    c.appendChild(b);
  });
}

function rsiColor(v){
  if(v>70) return "#ef4444";
  if(v<30) return "#10b981";
  return "#00d4ff";
}
function rsiSignal(v){
  if(v>70) return "🔴 Overbought";
  if(v<30) return "🟢 Oversold";
  return "🟡 Neutral";
}
function mcapBadge(cls){
  const map={"Large Cap":"badge-large","Mid Cap":"badge-mid","Small Cap":"badge-small"};
  return `<span class="badge ${map[cls]||'badge-large'}">${cls}</span>`;
}

function destroyCharts(){
  [priceChart,rsiChart,volChart].forEach(c=>{ if(c) c.destroy(); });
}

function buildCharts(d){
  destroyCharts();
  const chartOpts = { responsive:true, maintainAspectRatio:false,
    plugins:{legend:{labels:{color:"#64748b",font:{family:"DM Mono",size:10}}}},
    scales:{
      x:{ticks:{color:"#64748b",maxTicksLimit:8,font:{family:"DM Mono",size:10}},
         grid:{color:"rgba(30,45,69,.5)"}},
      y:{ticks:{color:"#64748b",font:{family:"DM Mono",size:10}},
         grid:{color:"rgba(30,45,69,.5)"}}
    }};

  // Price + MAs
  priceChart = new Chart(document.getElementById("priceChart"),{
    type:"line", data:{
      labels: d.chart_dates,
      datasets:[
        { label:"Price", data:d.chart_prices, borderColor:"#00d4ff",
          backgroundColor:"rgba(0,212,255,.06)", fill:true,
          borderWidth:2, pointRadius:0, tension:.3 },
        { label:"SMA20", data:d.chart_sma20, borderColor:"#f59e0b",
          borderWidth:1.5, borderDash:[4,4], pointRadius:0, fill:false },
        { label:"SMA50", data:d.chart_sma50, borderColor:"#7c3aed",
          borderWidth:1.5, borderDash:[4,4], pointRadius:0, fill:false },
      ]
    }, options:{...chartOpts}
  });

  // RSI
  rsiChart = new Chart(document.getElementById("rsiChart"),{
    type:"line", data:{
      labels: d.rsi_dates,
      datasets:[{ label:"RSI(14)", data:d.rsi_values,
        borderColor:"#a78bfa", borderWidth:2, pointRadius:0,
        backgroundColor:"rgba(167,139,250,.08)", fill:true, tension:.3 }]
    }, options:{ ...chartOpts,
      plugins:{ ...chartOpts.plugins,
        annotation:{ annotations:{
          ob:{ type:"line", yMin:70, yMax:70, borderColor:"#ef4444", borderWidth:1, borderDash:[4,4] },
          os:{ type:"line", yMin:30, yMax:30, borderColor:"#10b981", borderWidth:1, borderDash:[4,4] }
        }}},
      scales:{ ...chartOpts.scales, y:{ ...chartOpts.scales.y, min:0, max:100 } }
    }
  });

  // Volume
  volChart = new Chart(document.getElementById("volChart"),{
    type:"bar", data:{
      labels: d.chart_dates,
      datasets:[{ label:"Volume (M)", data:d.chart_vols,
        backgroundColor: d.chart_prices.map((p,i)=>
          i===0||p>=d.chart_prices[i-1]?"rgba(16,185,129,.7)":"rgba(239,68,68,.7)"),
        borderRadius:2 }]
    }, options:{...chartOpts,
      plugins:{legend:{display:false}}}
  });
}

function render(d){
  if(d.error){ document.getElementById("app").innerHTML=`<div class="loader"><p>⚠️ ${d.error}</p></div>`; return; }
  const up = d.change>=0;
  const chgHtml = `<span class="hero-chg ${up?"up":"dn"}">${up?"▲":"▼"} ₹${Math.abs(d.change)} (${up?"+":""}${d.change_pct}%)</span>`;
  document.getElementById("fxRate").textContent = `1 USD = ₹${d.fx_rate}`;

  const rsiNum = typeof d.rsi==="number"?d.rsi:0;
  const rsiW   = Math.min(100,Math.max(0,rsiNum));

  document.getElementById("app").innerHTML = `
  <div class="main">

  <!-- HERO -->
  <div class="hero">
    <div class="hero-left">
      <div class="company-name">${d.name}</div>
      <div class="symbol">${d.symbol} &nbsp;|&nbsp; ${d.sector} &nbsp;|&nbsp; ${mcapBadge(d.mcap_cls)}</div>
    </div>
    <div style="text-align:right">
      <div class="hero-price">₹${d.price.toLocaleString("en-IN")}</div>
      ${chgHtml}
    </div>
    <div class="hero-stats">
      <div class="hero-stat"><label>Day High</label><span>₹${d.high}</span></div>
      <div class="hero-stat"><label>Day Low</label><span>₹${d.low}</span></div>
      <div class="hero-stat"><label>Prev Close</label><span>₹${d.prev_close}</span></div>
      <div class="hero-stat"><label>Market Cap</label><span>₹${d.mcap_cr} Cr</span></div>
    </div>
  </div>

  <!-- SECTION TABS -->
  <div style="grid-column:1/-1">
    <div class="sec-tabs">
      <button class="sec-tab active" onclick="switchTab('overview')">📊 Overview</button>
      <button class="sec-tab" onclick="switchTab('financials')">💰 Financials</button>
      <button class="sec-tab" onclick="switchTab('balance')">🏦 Balance Sheet</button>
      <button class="sec-tab" onclick="switchTab('shareholding')">🥧 Shareholding</button>
      <button class="sec-tab" onclick="switchTab('actions')">📋 Corporate Actions</button>
    </div>
  </div>

  <!-- ═══════════ OVERVIEW TAB ═══════════ -->
  <div class="sec-panel active" id="panel-overview">

  <!-- PRICE CHART -->
  <div class="card span2">
    <div class="card-title">📈 Price Trend + Moving Averages (60D)</div>
    <div class="chart-wrap"><canvas id="priceChart"></canvas></div>
  </div>

  <!-- KEY METRICS -->
  <div class="card">
    <div class="card-title">📊 Key Metrics</div>
    <div class="metrics">
      <div class="metric"><label>PE Ratio (TTM)</label><span>${d.pe}</span></div>
      <div class="metric"><label>PB Ratio</label><span>${d.pb}</span></div>
      <div class="metric"><label>EPS (TTM ₹)</label><span>${d.eps}</span></div>
      <div class="metric"><label>Beta</label><span>${d.beta}</span></div>
      <div class="metric"><label>Div Yield</label><span>${d.div_yield}%</span></div>
      <div class="metric"><label>Avg Volume</label><span>${d.avg_vol}M</span></div>
    </div>
  </div>

  <!-- RSI -->
  <div class="card">
    <div class="card-title">📐 RSI (14) — ${rsiSignal(rsiNum)}</div>
    <div style="font-size:2rem;font-family:'DM Mono',monospace;color:${rsiColor(rsiNum)};font-weight:700;margin:8px 0">${d.rsi}</div>
    <div class="rsi-bar"><div class="rsi-fill" style="width:${rsiW}%;background:${rsiColor(rsiNum)}"></div></div>
    <div class="rsi-labels"><span>0 Oversold</span><span>30 &nbsp;&nbsp; 70</span><span>Overbought 100</span></div>
    <div class="chart-wrap-sm" style="margin-top:14px"><canvas id="rsiChart"></canvas></div>
  </div>

  <!-- 52W TRACKER -->
  <div class="card">
    <div class="card-title">📅 52-Week Range</div>
    <div style="display:flex;justify-content:space-between;margin-bottom:4px">
      <span style="color:var(--red);font-family:'DM Mono',monospace;font-size:.85rem">Low ₹${d.l52}</span>
      <span style="color:var(--green);font-family:'DM Mono',monospace;font-size:.85rem">High ₹${d.h52}</span>
    </div>
    <div class="gauge-bar">
      <div class="gauge-fill" style="width:100%"></div>
      <div class="gauge-marker" style="left:${d.pct52}%"></div>
    </div>
    <div style="text-align:center;margin-top:6px;font-family:'DM Mono',monospace;font-size:.85rem;color:var(--accent)">
      ₹${d.price} &nbsp;|&nbsp; ${d.pct52}% above 52W low
    </div>
    <div style="margin-top:20px">
      <div class="card-title">📢 Dividends</div>
      <div class="div-row"><span style="color:var(--muted)">Yield</span><span style="font-family:'DM Mono',monospace;color:var(--green)">${d.div_yield}%</span></div>
      <div class="div-row"><span style="color:var(--muted)">Last Div/Share</span><span style="font-family:'DM Mono',monospace">₹${d.last_div}</span></div>
    </div>
  </div>

  <!-- VOLUME -->
  <div class="card">
    <div class="card-title">📊 Volume Analysis (60D)</div>
    <div class="chart-wrap"><canvas id="volChart"></canvas></div>
  </div>

  <!-- SCORECARD -->
  <div class="card">
    <div class="card-title">🏆 Scorecard</div>
    <div class="scorecard">
      <div class="score-item"><span class="score-label">Market Cap Class</span><span class="score-val ${d.mcap_cls==="Large Cap"?"score-high":d.mcap_cls==="Mid Cap"?"score-mid":"score-low"}">${d.mcap_cls}</span></div>
      <div class="score-item"><span class="score-label">RSI Signal</span><span class="score-val ${rsiNum>70?"score-low":rsiNum<30?"score-high":"score-mid"}">${rsiSignal(rsiNum)}</span></div>
      <div class="score-item"><span class="score-label">PE vs Sector</span><span class="score-val ${d.pe<20?"score-high":"score-low"}">${d.pe<20?"Attractive":"Premium"}</span></div>
      <div class="score-item"><span class="score-label">Beta (Risk)</span><span class="score-val ${d.beta<1?"score-high":d.beta<1.5?"score-mid":"score-low"}">${d.beta<1?"Low Risk":d.beta<1.5?"Medium":"High Risk"}</span></div>
      <div class="score-item"><span class="score-label">52W Position</span><span class="score-val ${d.pct52>70?"score-high":d.pct52>40?"score-mid":"score-low"}">${d.pct52>70?"Near High":d.pct52>40?"Mid Range":"Near Low"}</span></div>
      <div class="score-item"><span class="score-label">Sector</span><span class="score-val score-mid">${d.sector}</span></div>
    </div>
  </div>

  <!-- PEER TABLE -->
  <div class="card span2">
    <div class="card-title">🔍 IT Sector Peers</div>
    <table class="peer-table">
      <tr><th>Stock</th><th>1Y Return</th><th>PE</th><th>PB</th><th>Div Yield</th></tr>
      <tr><td>Infosys (INFY)</td><td class="dn-text">▼14.30%</td><td>19.22</td><td>5.34</td><td>3.47%</td></tr>
      <tr><td>TCS</td><td class="dn-text">▼23.41%</td><td>18.66</td><td>8.47</td><td>4.33%</td></tr>
      <tr><td>HCL Technologies</td><td class="dn-text">▼13.15%</td><td>20.90</td><td>4.99</td><td>4.21%</td></tr>
      <tr><td>Wipro</td><td class="dn-text">▼12.88%</td><td>16.19</td><td>2.59</td><td>5.39%</td></tr>
      <tr><td>Tech Mahindra</td><td class="up-text">▲6.25%</td><td>29.79</td><td>—</td><td>—</td></tr>
    </table>
  </div>

  <!-- COMPANY -->
  <div class="card span2">
    <div class="card-title">📄 Company Overview</div>
    <p class="summary-text">${d.summary}</p>
  </div>

  </div><!-- end overview panel -->

  <!-- ═══════════ FINANCIALS TAB ═══════════ -->
  <div class="sec-panel" id="panel-financials">
  <div class="card span2">
    <div class="card-title">📈 Quarterly Revenue (₹ Cr)</div>
    <div class="chart-wrap"><canvas id="revChart"></canvas></div>
  </div>
  <div class="card span2">
    <div class="card-title">💵 Quarterly Net Income (₹ Cr)</div>
    <div class="chart-wrap"><canvas id="niChart"></canvas></div>
  </div>
  <div class="card span2">
    <div class="card-title">📊 Quarterly EBITDA &amp; Gross Profit (₹ Cr)</div>
    <div class="chart-wrap"><canvas id="ebitdaChart"></canvas></div>
  </div>
  </div><!-- end financials panel -->

  <!-- ═══════════ BALANCE SHEET TAB ═══════════ -->
  <div class="sec-panel" id="panel-balance">
  <div class="card span2">
    <div class="card-title">🏦 Balance Sheet — Latest Annual (₹ Cr)</div>
    <div id="bsContent"><div class="mini-loader">⏳ Loading balance sheet...</div></div>
  </div>
  </div><!-- end balance panel -->

  <!-- ═══════════ SHAREHOLDING TAB ═══════════ -->
  <div class="sec-panel" id="panel-shareholding">
  <div class="card" style="grid-column:1/-1;max-width:500px;margin:0 auto">
    <div class="card-title">🥧 Shareholding Pattern</div>
    <div style="height:300px;position:relative"><canvas id="shChart"></canvas></div>
    <div id="shLegend" style="margin-top:16px;display:flex;gap:16px;flex-wrap:wrap;justify-content:center"></div>
  </div>
  </div><!-- end shareholding panel -->

  <!-- ═══════════ CORPORATE ACTIONS TAB ═══════════ -->
  <div class="sec-panel" id="panel-actions">
  <div class="card span2">
    <div class="card-title">📋 Corporate Actions History</div>
    <div id="caContent"><div class="mini-loader">⏳ Loading corporate actions...</div></div>
  </div>
  </div><!-- end actions panel -->

  </div>
  <div class="refresh-bar" id="refreshBar">Auto-refreshes every 30s</div>
  `;

  buildCharts(d);
  loadExtra(currentStock);
}

// ── Section tab switching ──
function switchTab(name){
  document.querySelectorAll(".sec-tab").forEach((b,i)=>{
    const tabs=["overview","financials","balance","shareholding","actions"];
    b.classList.toggle("active", tabs[i]===name);
  });
  document.querySelectorAll(".sec-panel").forEach(p=>{
    p.classList.toggle("active", p.id==="panel-"+name);
  });
}

// ── Extra data (financials, balance sheet, shareholding, corp actions) ──
let revChart, niChart, ebitdaChart, shChart;
async function loadExtra(stockName){
  try{
    const res  = await fetch("/api/extra?name="+encodeURIComponent(stockName));
    const ex   = await res.json();
    buildFinancialCharts(ex.financials || {});
    buildBalanceSheet(ex.balance_sheet || {});
    buildShareholding(ex.shareholding || {});
    buildCorpActions(ex.corp_actions || []);
  } catch(e){ console.warn("Extra data error:", e); }
}

function buildFinancialCharts(fin){
  [revChart,niChart,ebitdaChart].forEach(c=>{ if(c) c.destroy(); });
  if(!fin.quarters || !fin.quarters.length) return;
  const labels = fin.quarters.map(q=>q.slice(0,7));
  const fOpts = (title,color)=>({
    responsive:true, maintainAspectRatio:false,
    plugins:{legend:{display:false},title:{display:false}},
    scales:{
      x:{ticks:{color:"#64748b",font:{family:"DM Mono",size:10}},grid:{color:"rgba(30,45,69,.5)"}},
      y:{ticks:{color:"#64748b",font:{family:"DM Mono",size:10}},grid:{color:"rgba(30,45,69,.5)"}}
    }
  });
  const parseVals = arr=>(arr||[]).map(v=>{ const n=parseFloat((v||"").replace(/,/g,"")); return isNaN(n)?null:n; });

  revChart = new Chart(document.getElementById("revChart"),{
    type:"bar", data:{labels, datasets:[{
      label:"Revenue",data:parseVals(fin.revenue),
      backgroundColor:"rgba(0,212,255,.7)",borderRadius:4
    }]}, options: fOpts()
  });
  niChart = new Chart(document.getElementById("niChart"),{
    type:"bar", data:{labels, datasets:[{
      label:"Net Income",data:parseVals(fin.net_income),
      backgroundColor: parseVals(fin.net_income).map(v=>v>=0?"rgba(16,185,129,.7)":"rgba(239,68,68,.7)"),
      borderRadius:4
    }]}, options: fOpts()
  });
  ebitdaChart = new Chart(document.getElementById("ebitdaChart"),{
    type:"bar", data:{labels, datasets:[
      {label:"EBITDA",data:parseVals(fin.ebitda),backgroundColor:"rgba(124,58,237,.7)",borderRadius:4},
      {label:"Gross Profit",data:parseVals(fin.gross_profit),backgroundColor:"rgba(245,158,11,.7)",borderRadius:4}
    ]}, options:{...fOpts(), plugins:{legend:{display:true,labels:{color:"#94a3b8",font:{family:"DM Mono",size:10}}}}}
  });
}

function buildBalanceSheet(bs){
  const el = document.getElementById("bsContent");
  if(!el) return;
  const keys = Object.keys(bs);
  if(!keys.length){ el.innerHTML='<p style="color:var(--muted);padding:16px">No balance sheet data available.</p>'; return; }
  const rows = keys.map(k=>`
    <tr>
      <td>${k}</td>
      <td>₹ ${bs[k]} Cr</td>
    </tr>`).join("");
  el.innerHTML = `<table class="bs-table"><tbody>${rows}</tbody></table>`;
}

function buildShareholding(sh){
  if(shChart){ shChart.destroy(); shChart=null; }
  const el = document.getElementById("shChart");
  const leg = document.getElementById("shLegend");
  if(!el) return;
  const keys = Object.keys(sh);
  if(!keys.length){ el.parentElement.innerHTML='<p style="color:var(--muted);padding:16px">No shareholding data available.</p>'; return; }
  const vals   = keys.map(k=>sh[k]);
  const colors = ["#7c3aed","#00d4ff","#10b981","#f59e0b","#ef4444"];
  shChart = new Chart(el,{
    type:"doughnut",
    data:{ labels:keys, datasets:[{ data:vals, backgroundColor:colors.slice(0,keys.length),
      borderColor:"#111827", borderWidth:3, hoverOffset:8 }] },
    options:{ responsive:true, maintainAspectRatio:false, cutout:"62%",
      plugins:{ legend:{display:false},
        tooltip:{callbacks:{label:ctx=>`${ctx.label}: ${ctx.parsed.toFixed(2)}%`}} } }
  });
  leg.innerHTML = keys.map((k,i)=>`
    <div style="display:flex;align-items:center;gap:6px;font-size:.78rem">
      <div style="width:12px;height:12px;border-radius:50%;background:${colors[i]}"></div>
      <span style="color:var(--muted)">${k}</span>
      <span style="font-family:'DM Mono',monospace;color:var(--text)">${sh[k]}%</span>
    </div>`).join("");
}

function buildCorpActions(actions){
  const el = document.getElementById("caContent");
  if(!el) return;
  if(!actions.length){ el.innerHTML='<p style="color:var(--muted);padding:16px">No corporate actions found.</p>'; return; }
  const rows = actions.map(a=>{
    const cls = a.type==="Dividend"?"ca-div":a.type==="Split"?"ca-split":"ca-bonus";
    return `<tr>
      <td style="font-family:'DM Mono',monospace;color:var(--muted)">${a.date}</td>
      <td><span class="ca-badge ${cls}">${a.type}</span></td>
      <td style="font-family:'DM Mono',monospace;color:var(--green);font-weight:600">${a.value}</td>
    </tr>`;
  }).join("");
  el.innerHTML = `<table class="ca-table">
    <thead><tr><th>Date</th><th>Type</th><th>Value</th></tr></thead>
    <tbody>${rows}</tbody>
  </table>`;
}

async function loadData(){
  document.getElementById("app").innerHTML = `<div class="loader"><div class="spinner"></div><p>Loading ${currentStock}...</p></div>`;
  try{
    const res = await fetch("/api/stock?name="+encodeURIComponent(currentStock));
    const d   = await res.json();
    render(d);
  } catch(e){
    document.getElementById("app").innerHTML = `<div class="loader"><p>⚠️ Error: ${e}</p></div>`;
  }
}

buildTabs();
loadData();
setInterval(loadData, 30000); // refresh every 30s
</script>
</body></html>
"""

@app.route("/")
def index():
    from flask import render_template_string
    return render_template_string(HTML, stocks_json=json.dumps(STOCKS))

@app.route("/api/stock")
def api_stock():
    from flask import request, Response
    import math, re
    name = request.args.get("name", "INFOSYS")
    sym  = STOCKS.get(name, "INFY.NS")
    data = get_stock_data(sym)
    # Serialize with allow_nan=False would raise; instead dump allowing NaN
    # then regex-replace bare NaN/Infinity tokens with null
    raw = json.dumps(data)
    safe = re.sub(r'\bNaN\b|\bInfinity\b|-Infinity', 'null', raw)
    return Response(safe, mimetype="application/json")

@app.route("/api/extra")
def api_extra():
    from flask import request, Response
    import re
    name = request.args.get("name", "INFOSYS")
    sym  = STOCKS.get(name, "INFY.NS")
    t    = yf.Ticker(sym)
    inf  = t.info
    data = {
        "balance_sheet": _get_balance_sheet(t),
        "financials":    _get_financials(t),
        "shareholding":  _get_shareholding(inf),
        "corp_actions":  _get_corp_actions(t),
    }
    raw  = json.dumps(data)
    safe = re.sub(r'\bNaN\b|\bInfinity\b|-Infinity', 'null', raw)
    return Response(safe, mimetype="application/json")

def run_flask():
    app.run(port=5050, use_reloader=False, debug=False)

t = threading.Thread(target=run_flask, daemon=True)
t.start()
import time; time.sleep(2)
print("✅ Flask server running on port 5050")


 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5050 is in use by another program. Either identify and stop that program, or start the server with a different port.


✅ Flask server running on port 5050


## Cell 4 — Get Your Live Public URL

In [8]:
from pyngrok import ngrok
import time

# Kill any old tunnels first
ngrok.kill()
time.sleep(2) # Add a small delay to allow ngrok to release the endpoint

tunnel = ngrok.connect(5050)
url = tunnel.public_url

print("=" * 60)
print("  YOUR LIVE DASHBOARD URL:")
print(f"  {url}")
print("=" * 60)
print("Open the URL above in any browser or share it!")
print("Dashboard auto-refreshes every 30 seconds.")
print("Run this cell again if the tunnel expires.")

  YOUR LIVE DASHBOARD URL:
  https://savor-cabdriver-grapple.ngrok-free.dev
Open the URL above in any browser or share it!
Dashboard auto-refreshes every 30 seconds.
Run this cell again if the tunnel expires.


## Cell 5 — Stop the Dashboard (Optional)

Run this when you are done to shut down the tunnel and server.

In [ ]:
from pyngrok import ngrok
ngrok.kill()
print("Tunnel closed.")